In [0]:
from pyspark.sql import functions as F

df_pib = spark.table("mvp_1.bronze.pib_municipio")

In [0]:
#Renomeia colunas para evitar problemas com caracteres especiais
df_pib = (
    df_pib
    .withColumnRenamed("Cód.", "codigo_municipio_ibge")
    .withColumnRenamed("Município", "municipio")
)

display(df_pib.limit(10))

codigo_municipio_ibge,municipio,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
1100015,Alta Floresta D'Oeste (RO),111291,143222,173991,167127,168805,191364,248962,256986,262077,280510,329029,341325,377799,421300,478217,485374,498980,495775,570242,734467,919520,1046343
1100023,Ariquemes (RO),449593,539636,657193,749021,790697,905203,1064822,1133095,1364694,1651885,1703642,1799853,1921532,2037799,2184346,2287910,2464704,2579278,2817331,3211294,3809355,4383815
1100031,Cabixi (RO),31768,40985,43392,49130,46884,49166,60588,69776,69611,77217,99586,96365,113477,116565,133342,138110,140503,139976,167153,238414,289783,300832
1100049,Cacoal (RO),474443,622437,622415,758960,743194,814890,928699,985479,1186494,1259024,1372705,1433254,1660650,1794478,1947283,2082761,2175840,2261930,2518845,2792506,3195489,3848468
1100056,Cerejeiras (RO),79174,99983,121366,129107,124415,143270,167474,190902,222021,260142,357333,353270,392417,397736,408194,439245,470647,506494,600630,743062,903099,1006389
1100064,Colorado do Oeste (RO),87254,103363,114815,126534,126670,137899,157026,174168,193093,206425,222071,242767,273759,285372,306518,328377,330232,334922,366922,424846,514219,580899
1100072,Corumbiara (RO),45165,57284,67309,70936,68935,72291,95990,119226,114768,145412,175872,168681,186868,190906,236624,332804,320416,349289,268353,396740,473845,509566
1100080,Costa Marques (RO),37308,50996,53905,62986,60812,74215,90971,99854,107583,120018,134091,149739,168605,187170,206101,212879,230151,238796,261944,316672,349796,413943
1100098,Espigão D'Oeste (RO),119312,153425,180676,203257,195271,220452,265616,291491,311788,347581,376177,426546,462389,483962,543318,566361,606072,624787,666321,773372,921382,1023973
1100106,Guajará-Mirim (RO),174680,247248,302287,327643,325665,374116,491320,521104,598167,715386,516172,605131,646154,682458,740352,775195,837459,892778,984614,1054255,1057883,1145034


In [0]:
#Remoção da última linha que não agrega informação relevante

df_pib = (
    df_pib
    .filter(
        F.col("codigo_municipio_ibge").isNotNull() &
        F.col("municipio").isNotNull()
    )
)

print(f"Municípios após remoção do registro: {df_pib.count()}")

Municípios após remoção do registro: 5570


In [0]:
#Separação do UF e Municipio para facilitação de leitura
df_pib = (
    df_pib
    .withColumn(
        "uf",
        F.regexp_extract(
            F.col("municipio"),
            r"\(([A-Z]{2})\)$",
            1
        )
    )
    .withColumn(
        "municipio",
        F.trim(
            F.regexp_replace(
                F.col("municipio"),
                r"\s*\([A-Z]{2}\)$",
                ""
            )
        )
    )
)

display(
    df_pib.select(
        "codigo_municipio_ibge",
        "municipio",
        "uf"
    ).limit(20)
)

codigo_municipio_ibge,municipio,uf
1100015,Alta Floresta D'Oeste,RO
1100023,Ariquemes,RO
1100031,Cabixi,RO
1100049,Cacoal,RO
1100056,Cerejeiras,RO
1100064,Colorado do Oeste,RO
1100072,Corumbiara,RO
1100080,Costa Marques,RO
1100098,Espigão D'Oeste,RO
1100106,Guajará-Mirim,RO


In [0]:
#validação do codigo
uf_ausente = (
    df_pib
    .filter(
        F.col("uf").isNull() |
        (F.col("uf") == "")
    )
    .count()
)

print(f"Municípios sem UF: {uf_ausente}")

Municípios sem UF: 0


In [0]:
# Padronização dos tipos de dados, temos string e numero na tabela

colunas_anos = [c for c in df_pib.columns if c.isdigit()]


for ano in colunas_anos:
    df_pib = df_pib.withColumn(
        ano,
        F.col(ano).cast("double")
    )

In [0]:
df_pib.select(colunas_anos).printSchema()

root
 |-- 2002: double (nullable = true)
 |-- 2003: double (nullable = true)
 |-- 2004: double (nullable = true)
 |-- 2005: double (nullable = true)
 |-- 2006: double (nullable = true)
 |-- 2007: double (nullable = true)
 |-- 2008: double (nullable = true)
 |-- 2009: double (nullable = true)
 |-- 2010: double (nullable = true)
 |-- 2011: double (nullable = true)
 |-- 2012: double (nullable = true)
 |-- 2013: double (nullable = true)
 |-- 2014: double (nullable = true)
 |-- 2015: double (nullable = true)
 |-- 2016: double (nullable = true)
 |-- 2017: double (nullable = true)
 |-- 2018: double (nullable = true)
 |-- 2019: double (nullable = true)
 |-- 2020: double (nullable = true)
 |-- 2021: double (nullable = true)
 |-- 2022: double (nullable = true)
 |-- 2023: double (nullable = true)



[CAST_INVALID_INPUT] The value '...' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from , line 9 in cell [18]

Tentando descobrir o erro


In [0]:
# Identifica as colunas correspondentes aos anos
colunas_anos = [c for c in df_pib.columns if c.isdigit()]

# Verifica a presença de valores "..." em cada ano
for ano in colunas_anos:
    qtd = (
        df_pib
        .filter(
            F.trim(F.col(ano).cast("string")) == "..."
        )
        .count()
    )

    if qtd > 0:
        print(f"{ano}: {qtd} valores '...'")

Foram identificados string no meio de alguns periodos, por isso originalmente essas colunas vieram como strings, vamos fazer a mudança para NULL


In [0]:
# Substitui "..." por NULL e padroniza os valores de PIB como double

for ano in colunas_anos:
    df_pib = df_pib.withColumn(
        ano,
        F.when(
            F.trim(F.col(ano).cast("string")) == "...",
            F.lit(None)
        )
        .otherwise(F.col(ano))
        .cast("double")
    )

    df_pib.select(colunas_anos).printSchema()

root
 |-- 2002: double (nullable = true)
 |-- 2003: double (nullable = true)
 |-- 2004: double (nullable = true)
 |-- 2005: double (nullable = true)
 |-- 2006: double (nullable = true)
 |-- 2007: double (nullable = true)
 |-- 2008: double (nullable = true)
 |-- 2009: double (nullable = true)
 |-- 2010: double (nullable = true)
 |-- 2011: double (nullable = true)
 |-- 2012: double (nullable = true)
 |-- 2013: double (nullable = true)
 |-- 2014: double (nullable = true)
 |-- 2015: double (nullable = true)
 |-- 2016: double (nullable = true)
 |-- 2017: double (nullable = true)
 |-- 2018: double (nullable = true)
 |-- 2019: double (nullable = true)
 |-- 2020: double (nullable = true)
 |-- 2021: double (nullable = true)
 |-- 2022: double (nullable = true)
 |-- 2023: double (nullable = true)

root
 |-- 2002: double (nullable = true)
 |-- 2003: double (nullable = true)
 |-- 2004: double (nullable = true)
 |-- 2005: double (nullable = true)
 |-- 2006: double (nullable = true)
 |-- 2007: double

In [0]:
# Transformação de coluna para linha
df_pib_silver = (
    df_pib
    .unpivot(
        ids=["codigo_municipio_ibge", "municipio", "uf"],
        values=colunas_anos,
        variableColumnName="ano",
        valueColumnName="pib_mil_reais"
    )
    .withColumn("ano", F.col("ano").cast("int"))
)

display(df_pib_silver.limit(30))

codigo_municipio_ibge,municipio,uf,ano,pib_mil_reais
1100015,Alta Floresta D'Oeste,RO,2002,111291.0
1100023,Ariquemes,RO,2002,449593.0
1100031,Cabixi,RO,2002,31768.0
1100049,Cacoal,RO,2002,474443.0
1100056,Cerejeiras,RO,2002,79174.0
1100064,Colorado do Oeste,RO,2002,87254.0
1100072,Corumbiara,RO,2002,45165.0
1100080,Costa Marques,RO,2002,37308.0
1100098,Espigão D'Oeste,RO,2002,119312.0
1100106,Guajará-Mirim,RO,2002,174680.0


In [0]:
# Validação da tabela de PIB após transformaçã

total_registros = df_pib_silver.count()

municipios = (
    df_pib_silver
    .select("codigo_municipio_ibge")
    .distinct()
    .count()
)

duplicados = (
    df_pib_silver
    .groupBy("codigo_municipio_ibge", "ano")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

pib_ausente = (
    df_pib_silver
    .filter(F.col("pib_mil_reais").isNull())
    .count()
)

periodo = (
    df_pib_silver
    .agg(
        F.min("ano").alias("inicio"),
        F.max("ano").alias("fim")
    )
    .first()
)

print(f"Total de registros: {total_registros}")
print(f"Municípios distintos: {municipios}")
print(f"Duplicidades Município/Ano: {duplicados}")
print(f"Valores de PIB ausentes: {pib_ausente}")
print(f"Período: {periodo['inicio']}-{periodo['fim']}")

Total de registros: 122540
Municípios distintos: 5570
Duplicidades Município/Ano: 0
Valores de PIB ausentes: 74
Período: 2002-2023


In [0]:
# Salva a tabela tratada na camada Silver

(
    df_pib_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_1.silver.pib_municipio")
)

print("Tabela mvp_1.silver.pib_municipio salva com sucesso.")

Tabela mvp_1.silver.pib_municipio salva com sucesso.
